# 1. Integration of 1 million (RGS) * 36 times (RDS) random sampling results to obtain the importance score of genes in a single gene set within a single dataset
> Input: Clustering performance evaluation metrics for each RDS from 1 million iterations

> Intermediate processing steps:

> (1) For each indicator in each RDS, sort from largest to smallest, select the top % RGS, and consider them effective in the primary and metastatic classification. Mark the corresponding matrix values as 1, and set the other RGS as 0, generating a 0-1 matrix.

> (2) Organize the same indicator from each RDS into a matrix; assess the similarity of matrices from different indicators in the same dataset. If the matrices are similar, merge them (matrix addition—if any matrix indicator has a value of 1, the corresponding position in the merged matrix will also be 1); if the matrices are not similar, keep them separately.

> (3) For each dataset, select the RGS that can effectively differentiate primary and metastatic samples in 90% of the RDS from the 0-1 matrix for each indicator. Then, perform a frequency count of these selected RGS across all indicators in the dataset. This frequency represents the gene importance score. Sort the genes by their importance scores from highest to lowest.

> Output: Gene ranking for each dataset's 25 gene sets, along with gene importance rankings.

In [ ]:
Top_10_RGS <- function(df) { 
  n_rows <- nrow(df)

  result_matrix <- apply(df, 2, function(col) {  
    top_n <- ceiling(0.1 * n_rows)
    idx <- order(col, decreasing = TRUE)[1:top_n]
    out <- integer(n_rows)
    out[idx] <- 1
    out
  })
  result_matrix <- as.matrix(result_matrix)
  rownames(result_matrix) <- rownames(df)
  colnames(result_matrix) <- colnames(df)
  return(result_matrix)
}

In [ ]:
geneList = c('Cluster1','Cluster2','Cluster3','Cluster4','Cluster5','Cluster6','Cluster7','Cluster8','Cluster9','Cluster10',
             'Cluster11','Cluster12','Cluster13','Cluster14','Cluster15','Cluster16','Cluster17','Cluster18','Cluster19','Cluster20','H1','H2','H3','H4','H5')

dataList = c('GSE125989','GSE60542','GSE62321','GSE73168','GSE85258','GSE44408') 

ALLGeneOrder = list()

for (g in geneList){
    cat("==== Processing gene set:", g, "====\n")  
    ALLGeneOrder[[g]] = list()  
    
    for (d in dataList){
        
        Benchmark <- paste0("./Project/100wSampling/",g,"/",d,"/Benchmark.csv")
        GenesSampledRes <- paste0("./Project/100wSampling/",g,"/",d,"/GenesSampledRes.csv")
        
        if (!file.exists(Benchmark) | !file.exists(GenesSampledRes)) {
            message("Skip (file missing): ", g, " / ", d)
            next
        }
        RDS_4index <- tryCatch(read.csv(Benchmark, row.names = 1),error = function(e) NULL)  # Read performance metrics                
        RGSlist <- tryCatch(read.csv(GenesSampledRes, row.names = 1),error = function(e) NULL) # Read the randomly sampled gene set RDS list
        if (is.null(RDS_4index) | is.null(RGSlist) |
            nrow(RDS_4index) == 0 | nrow(RGSlist) == 0) {
            message("Skip (empty or read error): ", g, " / ", d)
            next
        }

        # (1) For each dataset and gene set, build 4 unit matrices for the 4 performance metrics
        ACC <- RDS_4index[, grep("acc", colnames(RDS_4index), value = TRUE)]
        AUC <- RDS_4index[, grep("auc", colnames(RDS_4index), value = TRUE)]
        MCC <- RDS_4index[, grep("mcc", colnames(RDS_4index), value = TRUE)]
        Kappa <- RDS_4index[, grep("kappa", colnames(RDS_4index), value = TRUE)]
        # Select the top 10% of RGS in each metric as the effective RGS and build unit matrices
        ACC_matrix <- Top_10_RGS(ACC)
        ACC_matrix <- Top_10_RGS(ACC)
        AUC_matrix <- Top_10_RGS(AUC)
        MCC_matrix <- Top_10_RGS(MCC)
        Kappa_matrix <- Top_10_RGS(Kappa)

        # (2) Integrate the results of multiple metrics to construct the gene set's importance ranking in a bulk dataset
        # Combine the unit matrices from the 4 metrics, where an RGS is considered effective if any metric indicates it has distinguishing ability
        all_index = ACC_matrix + AUC_matrix + MCC_matrix + Kappa_matrix
        all_index = ACC_matrix+AUC_matrix+MCC_matrix+Kappa_matrix
        all_index[all_index > 1] <- 1 
        # Select RGS that are effective in more than 90% of RDS from all metrics
        selected_RGS <- rownames(all_index)[rowMeans(all_index) > 0.9]
        # Retrieve the specific genes for the selected RGS and merge them into a single gene set
        GeneSet = c(unlist(RGSlist[rownames(RGSlist) %in% selected_RGS,], use.names = FALSE))

        if (length(GeneSet) == 0) {
          ALLGeneOrder[[g]][[d]] <- numeric(0)
          next
        }
        # Count the frequency of each gene's occurrence as its importance score, and then sort the gene list by the score
        geneOrder <- names(sort(table(GeneSet), decreasing = TRUE)) 

        ALLGeneOrder[[g]][[d]] = geneOrder
    }
}

# 2. Integration of Gene Ranks from Multiple Datasets to Obtain a Comprehensive Gene Importance Ranking

> Input: 25 gene sets with importance rankings from multiple datasets

> Output: A comprehensive gene set with 25 genes ranked by importance

In [ ]:
source("./Function/2.2.RankAggregate.RRA.R")

In [ ]:
setList = list()
for(g in names(ALLGeneOrder)){
    setList[[g]] = RankAggregate.RRA (ALLGeneOrder[[g]],unlist(geneSet[[g]]))
}

# 3. Identify the Optimal Gene Set Subset for Classification in Lymph Node Metastasis Data from Breast Cancer

> Input: ① A comprehensive gene set of 25 genes ranked by importance; ② Lymph node metastasis data from breast cancer

> Intermediate processing steps:

> (1) Forward feature selection method: Sequentially use the top n genes from each gene set as classifier features and predict sample categories using SVM and LR classifiers. Calculate classification performance metrics for different feature sets.

> (2) Based on classification performance metrics, select the top % genes from each gene set as the final feature gene set.

> Output: A multi-feature gene set that can be used to identify metastasis potential subpopulations in single-cell data

In [ ]:
source("./Function/2.3.4.EvaluateSignature.R")

In [ ]:
setSelect = list() 
for (g in names(setList) ){
    cat("==== Processing gene set:", g, "====\n")
    set = setList[[g]]
    setSelect[[g]]<-Classifier(expr=GSE44408_exp, pData=pData44408,genelist=set$Name)
}

# 4. Complete-selection label-permutation analysis

This section evaluates selection-stage false positives by repeating the complete RGS search and selection procedure under 500 randomly permuted sample labels. Analyses are performed separately for each paired bulk dataset.

**Datasets:** GSE125989, GSE60542, GSE62321, GSE73168, GSE85258, and GSE44408. GSE62321 is shown as an example; repeat the analysis sequentially for the remaining datasets.

**Language:** Python is used only for GPU-accelerated fuzzy clustering and benchmark generation (Section 4.1). R is used for RGS selection, statistical summaries, and visualization (Sections 4.2–4.4).

## 4.1. Benchmark generation (Python; GPU)

In [ ]:
import torch
import os
import random
import numpy as np
import pandas as pd
import gc
import pyreadr

os.chdir('./Project')

os.makedirs("./100wSampling", exist_ok=True)

for i in range(1,21):
    if not os.path.exists(f'./100wSampling/Cluster{i}'):
        os.mkdir(f'./100wSampling/Cluster{i}')

for i in range(1,6):
    if not os.path.exists(f'./100wSampling/H{i}'):
        os.mkdir(f'./100wSampling/H{i}')

def FullyReset(SEED=42):

    os.environ["PYTHONHASHSEED"] = str(SEED)
    random.seed(SEED)
    np.random.seed(SEED)

    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

GOterms = pd.read_csv('GOterms_long.csv')
print(GOterms.head())
print("GO terms loaded")

GOdict = {}
for i in sorted(set(GOterms['Cluster'])):
    GOdict[i] = np.array(GOterms['GO_term'][GOterms['Cluster'] == i]).tolist()
    print(i, len(GOdict[i]))
print("GO term groups prepared")

gse_id = '62321'  # Change to another dataset ID for sequential runs 
label = 'group' 
result = pyreadr.read_r(f'./GSE{gse_id}_exp.Rdata')
bulk_data = result[f'GSE{gse_id}_exp']
bulk_data = bulk_data.iloc[:,1:].T
bulk_data.columns = result[f'GSE{gse_id}_exp']['symbol']
bulk_data.head()
print("Bulk expression loaded")

result = pyreadr.read_r(f'./pData{gse_id}.Rdata')
bulk_label = result[f'pData{gse_id}']
num_samples = len(bulk_label)
num_paires = int(num_samples / 2)

print("Bulk samples indexed")

sub_GOdict = {}
for i in sorted(GOdict.keys()):
    overlop_genes = [i for i in set(GOdict[i]).intersection(bulk_data.columns)]
    if len(overlop_genes) >= 30:
        sub_GOdict[i] = sorted(overlop_genes)
        print(f"{i}:\t{len(sub_GOdict[i])}")
print("Signature gene overlap prepared")

class BatchFuzzyCMeans:
    def __init__(self, n_clusters=2, m=2.0, max_iter=100, tol=1e-4, device="cuda"):
        self.c = n_clusters
        self.m = m
        self.max_iter = max_iter
        self.tol = tol
        self.device = device

    def fit(self, X, y_true):
        if isinstance(X, torch.Tensor):
            X = X.to(self.device, dtype=torch.float32)
        else:
            X = torch.from_numpy(X).to(self.device, dtype=torch.float32)
        batch, n, d = X.shape

        U = torch.rand(batch, n, self.c, device=self.device)
        U = U / U.sum(dim=2, keepdim=True)

        for _ in range(self.max_iter):

            U_old = U.clone()

            um = U ** self.m
            centers = (um.transpose(1, 2) @ X) / um.sum(dim=1, keepdim=True).transpose(1,2)

            dist = torch.cdist(X, centers)
            dist = torch.clamp(dist, min=1e-8)

            power = 2.0 / (self.m - 1)

            inv_dist = dist.pow(-power)
            U = inv_dist / inv_dist.sum(dim=2, keepdim=True)

            if torch.max(torch.abs(U - U_old)) < self.tol:
                break

        self.centers = centers
        self.U_ = U
        self.Class_ = self.U_.argmax(dim=2)
        self.match_label_cluster(y_true)
        self.batch_benchmark(y_true)

    def match_label_cluster(self, y_true):
        n = y_true.size(1)
        pred = self.Class_

        tp = ((y_true == 1) & (pred == 1)).sum(dim=1).float()
        tn = ((y_true == 0) & (pred == 0)).sum(dim=1).float()
        fp = ((y_true == 0) & (pred == 1)).sum(dim=1).float()
        fn = ((y_true == 1) & (pred == 0)).sum(dim=1).float()

        po = (tp + tn) / n
        p_yes_true = (tp + fn) / n
        p_yes_pred = (tp + fp) / n
        p_no_true  = (tn + fp) / n
        p_no_pred  = (tn + fn) / n

        pe = p_yes_true * p_yes_pred + p_no_true * p_no_pred

        kappa = (po - pe) / (1 - pe + 1e-12)

        self.kappa_raw = kappa.clone()

        mask = kappa < 0

        self.U = self.U_.clone()

        self.U[mask] = self.U_[mask].flip(-1)

        self.Class = self.U.argmax(dim=2)

        self.kappa = kappa.abs()

        return self
    
    def batch_benchmark(self, y_true):
        tp = ((y_true == 1) & (self.Class == 1)).sum(dim=1).float()
        tn = ((y_true == 0) & (self.Class == 0)).sum(dim=1).float()
        fp = ((y_true == 0) & (self.Class == 1)).sum(dim=1).float()
        fn = ((y_true == 1) & (self.Class == 0)).sum(dim=1).float()
        
        tpr = tp / (tp + fn + 1e-12)
        tnr = tn / (tn + fp + 1e-12)
        self.acc = 0.5 * (tpr + tnr)
        
        numerator = tp * tn - fp * fn
        denominator = torch.sqrt(
            (tp + fp) * (tp + fn) *
            (tn + fp) * (tn + fn)
        ) + 1e-12
        self.mcc = numerator / denominator

        B, N = y_true.shape
        y_score = self.U[:, :, 1]
        order = torch.argsort(y_score, dim=1)
        y_sorted = torch.gather(y_true, 1, order)

        n_pos = (y_true == 1).sum(dim=1).float()
        n_neg = (y_true == 0).sum(dim=1).float()

        cum_neg = torch.cumsum((y_sorted == 0).float(), dim=1)

        auc = (cum_neg * (y_sorted == 1).float()).sum(dim=1)

        self.auc = auc / (n_pos * n_neg + 1e-12)
        
        return self 




bulk_label['pseudo_group'] = bulk_label[label].to_numpy()
sorted_label =  bulk_label.sort_values(['pseudo_group', 'Num'])
sorted_label['binary_label'] = [0 if j in ['primary', 'PT', 'tumor', 'Primary'] else 1 for j in sorted_label.loc[:, 'pseudo_group']]
sorted_data = bulk_data.loc[sorted_label.iloc[:,0],:]
assert np.all(sorted_data.index == sorted_label.iloc[:,0])
print("Bulk samples indexed")
g1_label = sorted_label[0:int(num_samples/2)]
g2_label = sorted_label[int(num_samples/2):]

FullyReset(1)
for cluster in sorted(sub_GOdict.keys()):

# cluster=[i for i in sub_GOdict.keys()][count]

    print(cluster)
    genes_length = len(sub_GOdict[cluster])
    fair_probs = torch.ones(genes_length, device='cuda', dtype=torch.float32) / genes_length

    T_genes = 1000000
    k_genes = 30

    samples1 = torch.multinomial(
        fair_probs.expand(T_genes, -1),
        k_genes,
        replacement=False,
    )
    
    samples = samples1.cpu().numpy()
    sampled_genes = {
        f"rall_times:{i+1}": [sub_GOdict[cluster][j] for j in row]
        for i, row in enumerate(samples)
    }
    sampledgenes_mat = pd.DataFrame(sampled_genes).T
    print(sampledgenes_mat.head())
    sampledgenes_mat.to_csv(f'./100wSampling/{cluster}/GSE{gse_id}_RealGenesSampledRes.csv')
    
    T_samples = 36
    k_samples = int(0.7 * num_paires)

    fair_probs = torch.ones(num_paires) / num_paires

    samples2 = torch.multinomial(
        fair_probs.expand(T_samples, -1),
        k_samples,
        replacement=False,
    )

    bfcm =  BatchFuzzyCMeans()
    class_samples2 = torch.concat((samples2, samples2+num_paires),dim=1)
    idx = torch.argsort(torch.rand(class_samples2.shape), dim=1)
    class_samples2_shuffle = torch.gather(class_samples2, dim=1, index=idx)


    acc_list = []
    auc_list = []
    mcc_list = []
    kappa_list = []
    chunk=10000
    for start in range(0, T_genes, chunk):
        end = start + chunk
        samples_chunk = samples[start:end]
        label_graph = np.squeeze(sorted_label.loc[:, 'binary_label'].values.reshape((-1, 1)) [class_samples2_shuffle, :], axis=-1)
    
        data_graph = sorted_data.values[
                class_samples2_shuffle, :
            ][:, :, samples_chunk]

        newdata = torch.from_numpy(data_graph).to(
            "cuda",
            dtype=torch.float32,
            non_blocking=True,
        )
        
        newdata = newdata.permute(0, 2, 1, 3)
        newdata = newdata.reshape(-1, k_samples*2, k_genes)
        newlabel = torch.tensor(label_graph, device='cuda',dtype=torch.int).unsqueeze(1).expand(T_samples, len(samples_chunk), k_samples*2)# [36, 1w, 0.7m]
        newlabel = newlabel.reshape(-1, k_samples*2)
        bfcm.fit(newdata, newlabel)
        acc_list.append(
            bfcm.acc.cpu().reshape(T_samples, -1)
        )
        
        auc_list.append(
            bfcm.auc.cpu().reshape(T_samples, -1)
        )
        
        mcc_list.append(
            bfcm.mcc.cpu().reshape(T_samples, -1)
        )
        
        kappa_list.append(
            bfcm.kappa.cpu().reshape(T_samples, -1)
        )
        print(torch.cuda.memory_allocated()/1024**3)    
    acc = torch.cat(acc_list, dim=1).reshape(-1)
    auc = torch.cat(auc_list, dim=1).reshape(-1)
    mcc = torch.cat(mcc_list, dim=1).reshape(-1)
    kappa = torch.cat(kappa_list, dim=1).reshape(-1)
    benchmark_res = pd.DataFrame({
        "acc": acc.numpy(),
        "auc": auc.numpy(),
        "mcc": mcc.numpy(),
        "kappa": kappa.numpy(),
    })

    benchmark_res.to_parquet(f'./100wSampling/{cluster}/GSE{gse_id}_RealLabelBenchmark.parquet')
    del newdata
    del newlabel
    del bfcm
    gc.collect()
    torch.cuda.empty_cache()
print("True-label benchmarks completed")


rng = np.random.default_rng(42)
for epoch in range(0, 500):
    bulk_label['pseudo_group'] = rng.permutation(bulk_label[label].to_numpy())
    sorted_label =  bulk_label.sort_values(['pseudo_group', 'Num'])
    sorted_label['binary_label'] = [0 if j in ['primary', 'PT', 'tumor', 'Primary'] else 1 for j in sorted_label.loc[:, 'pseudo_group']]
    sorted_data = bulk_data.loc[sorted_label.iloc[:,0],:]
    assert np.all(sorted_data.index == sorted_label.iloc[:,0])
    g1_label = sorted_label[0:int(num_samples/2)]
    g2_label = sorted_label[int(num_samples/2):]
    
    FullyReset(epoch + 1001)
    for cluster in sorted(sub_GOdict.keys()):

    # cluster=[i for i in sub_GOdict.keys()][count]

        print(cluster)
        genes_length = len(sub_GOdict[cluster])
        fair_probs = torch.ones(genes_length, device='cuda', dtype=torch.float32) / genes_length

        T_genes = 1000000
        k_genes = 30

        # Reuse the true-label candidate RGS library for exact index comparison.
        saved = pd.read_csv(f'./100wSampling/{cluster}/GSE{gse_id}_RealGenesSampledRes.csv', index_col=0)
        if len(saved) != T_genes:
            raise ValueError("Candidate library length differs from T_genes")
        mapping = {v:i for i,v in enumerate(sub_GOdict[cluster])}
        samples = saved.apply(lambda col: col.map(mapping)).to_numpy(dtype=np.int64)
        if np.any(samples < 0):
            raise ValueError("Unmapped gene in candidate RGS library")
        T_samples = 36
        k_samples = int(0.7 * num_paires)

        fair_probs = torch.ones(num_paires) / num_paires

        samples2 = torch.multinomial(
            fair_probs.expand(T_samples, -1),
            k_samples,
            replacement=False,
        )

        bfcm =  BatchFuzzyCMeans()
        class_samples2 = torch.concat((samples2, samples2+num_paires),dim=1)
        idx = torch.argsort(torch.rand(class_samples2.shape), dim=1)
        class_samples2_shuffle = torch.gather(class_samples2, dim=1, index=idx)


        acc_list = []
        auc_list = []
        mcc_list = []
        kappa_list = []
        chunk=25000
        for start in range(0, T_genes, chunk):
            end = start + chunk
            samples_chunk = samples[start:end]
            label_graph = np.squeeze(sorted_label.loc[:, 'binary_label'].values.reshape((-1, 1)) [class_samples2_shuffle, :], axis=-1)
        
            data_graph = sorted_data.values[
                    class_samples2_shuffle, :
                ][:, :, samples_chunk]

            newdata = torch.from_numpy(data_graph).to(
                "cuda",
                dtype=torch.float32,
                non_blocking=True,
            )
            
            newdata = newdata.permute(0, 2, 1, 3)
            newdata = newdata.reshape(-1, k_samples*2, k_genes)
            newlabel = torch.tensor(label_graph, device='cuda',dtype=torch.int).unsqueeze(1).expand(T_samples, len(samples_chunk), k_samples*2)# [36, 1w, 0.7m]
            newlabel = newlabel.reshape(-1, k_samples*2)
            bfcm.fit(newdata, newlabel)
            acc_list.append(
                bfcm.acc.cpu().reshape(T_samples, -1)
            )
            
            auc_list.append(
                bfcm.auc.cpu().reshape(T_samples, -1)
            )
            
            mcc_list.append(
                bfcm.mcc.cpu().reshape(T_samples, -1)
            )
            
            kappa_list.append(
                bfcm.kappa.cpu().reshape(T_samples, -1)
            )
            print(torch.cuda.memory_allocated()/1024**3)    
        acc = torch.cat(acc_list, dim=1).reshape(-1)
        auc = torch.cat(auc_list, dim=1).reshape(-1)
        mcc = torch.cat(mcc_list, dim=1).reshape(-1)
        kappa = torch.cat(kappa_list, dim=1).reshape(-1)
        benchmark_res = pd.DataFrame({
            "acc": acc.numpy(),
            "auc": auc.numpy(),
            "mcc": mcc.numpy(),
            "kappa": kappa.numpy(),
        })

        benchmark_res.to_parquet(f'./100wSampling/{cluster}/GSE{gse_id}_ShuffleLabel_{epoch}_Benchmark.parquet')
        del newdata
        del newlabel
        del bfcm
        gc.collect()
        torch.cuda.empty_cache()
    print(f'Permutation {epoch+1} completed')



## 4.2. RGS selection under true and permuted labels (R)

The same selection criteria are applied independently to the true-label benchmark and each of the 500 permuted-label benchmarks. The RGS library is indexed consistently across runs.


In [ ]:
# R
dataset_list <- c("GSE125989", "GSE60542", "GSE62321","GSE73168", "GSE85258", "GSE44408")
dataset_name <- "GSE62321"
project_dir <- "./Project/100wSampling"
out_dir <- file.path(project_dir, "RGS_false_positive", dataset_name)
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)
geneList <- c(paste0("Cluster", 1:20), paste0("H", 1:5))
n_rgs <- 1000000L
n_perm <- 500L


In [ ]:
# Language: R
library(arrow)
Top_10_RGS <- function(df) {
  df <- as.matrix(df)
  result <- matrix(0L,nrow(df),ncol(df))
  for(j in seq_len(ncol(df))) {
    idx <- order(df[,j],decreasing=TRUE)[seq_len(ceiling(.1*nrow(df)))]
    result[idx,j] <- 1L
  }
  result
}
select_rgs <- function(path) {
  if(!file.exists(path)) stop("Missing: ",path)
  benchmark <- as.data.frame(read_parquet(path))
  metric_names <- c("acc","auc","mcc","kappa")
  if(!all(metric_names %in% names(benchmark))) stop("Missing metric: ",path)
  binary <- lapply(benchmark[metric_names],function(x) {
    if(length(x)!=n_rgs*36L) stop("Unexpected benchmark dimensions: ",path)
    Top_10_RGS(matrix(as.numeric(x),nrow=n_rgs,ncol=36L))
  })
  all_index <- Reduce(`|`,lapply(binary,function(x)x>0))
  which(rowMeans(all_index)>.9)
}
selected_RGS_index <- list()
ShuffleLabel_selected <- list()
for(g in geneList) {
  folder <- file.path(project_dir,g)
  true_path <- file.path(folder,paste0(dataset_name,"_RealLabelBenchmark.parquet"))
  if(!file.exists(true_path)) next
  selected_RGS_index[[g]] <- select_rgs(true_path)
  perm <- vector("list",n_perm)
  names(perm) <- paste0("ShuffleLabel_",0:(n_perm-1))
  for(p in 0:(n_perm-1)) {
    f <- file.path(folder,paste0(dataset_name,"_ShuffleLabel_",p,"_Benchmark.parquet"))
    perm[[p+1]] <- select_rgs(f)
  }
  ShuffleLabel_selected[[g]] <- perm
  message(g,": true=",length(selected_RGS_index[[g]]))
}
if(!length(selected_RGS_index)) stop("No signature results available")
save(selected_RGS_index,file=file.path(out_dir,"selected_RGS_index.RData"))
save(ShuffleLabel_selected,file=file.path(out_dir,"ShuffleLabel_selected_RGS_index.RData"))


## 4.3. Permutation-based retention statistics (R)

In [ ]:
# Language: R
counts <- list(); recurrences <- list()
for(g in names(selected_RGS_index)) {
  true <- unique(selected_RGS_index[[g]])
  perms <- lapply(ShuffleLabel_selected[[g]],unique)
  null <- vapply(perms,length,integer(1))
  counts[[g]] <- data.frame(Dataset=dataset_name,Signature=g,
    Permutation=seq_len(n_perm),N_RGS=null,N_true=length(true))
  if(length(true)) recurrences[[g]] <- data.frame(
    Dataset=dataset_name,Signature=g,RGS_index=true,
    N_selected_in_null=vapply(true,function(id)
      sum(vapply(perms,function(p)id %in% p,logical(1))),integer(1)))
}
Permutation_RGS_count <- do.call(rbind,counts)
Count_summary <- do.call(rbind,lapply(split(Permutation_RGS_count,
                                          Permutation_RGS_count$Signature),
 function(d) data.frame(Dataset=dataset_name,Signature=d$Signature[1],
  N_true=d$N_true[1],Null_mean=mean(d$N_RGS),
  Null_Q025=unname(quantile(d$N_RGS,.025)),
  Null_Q975=unname(quantile(d$N_RGS,.975)),
  Empirical_P=(1+sum(d$N_RGS>=d$N_true[1]))/(n_perm+1))))
Count_summary$FDR_BH <- p.adjust(Count_summary$Empirical_P,method="BH")
RGS_recurrence <- if(length(recurrences)) do.call(rbind,recurrences) else
  data.frame(Dataset=character(),Signature=character(),RGS_index=integer(),
             N_selected_in_null=integer())
RGS_recurrence$Null_selection_frequency <- RGS_recurrence$N_selected_in_null/n_perm
write.csv(Permutation_RGS_count,file.path(out_dir,"Permutation_RGS_count.csv"),row.names=FALSE)
write.csv(Count_summary,file.path(out_dir,"Count_summary.csv"),row.names=FALSE)
write.csv(RGS_recurrence,file.path(out_dir,"RGS_recurrence.csv"),row.names=FALSE)
print(Count_summary)


## 4.4. Visualization of permutation null distributions (R; ggplot2)

In [ ]:
# Language: R
library(ggplot2)
library(dplyr)
read_counts <- function(ds) {
  f <- file.path(project_dir,"RGS_false_positive",ds,"Permutation_RGS_count.csv")
  if(!file.exists(f)) return(NULL)
  d <- read.csv(f)
  if(length(unique(d$Permutation))!=500L) stop("Incomplete: ",ds)
  d
}
all_counts <- bind_rows(lapply(dataset_list,read_counts))
if(nrow(all_counts)) {
  null_total <- all_counts %>% group_by(Dataset,Permutation) %>%
    summarise(N_RGS=sum(N_RGS),.groups="drop")
  true_total <- all_counts %>% distinct(Dataset,Signature,N_true) %>%
    group_by(Dataset) %>% summarise(N_true=sum(N_true),.groups="drop")
  p_a <- ggplot(null_total,aes(N_RGS)) +
    geom_histogram(bins=30,fill="grey35",color="white") +
    geom_vline(data=true_total,aes(xintercept=N_true),
               color="red",linetype="dashed") +
    facet_wrap(~Dataset,ncol=3,scales="free_x") +
    labs(x="Number of retained RGSs",y="Number of label permutations",
         title="Complete-selection permutation null") + theme_bw()
  print(p_a)
  ggsave(file.path(project_dir,"RGS_false_positive","panel_a.pdf"),
         p_a,width=12,height=7)
}


In [ ]:
# Language: R
for(ds in dataset_list) {
  d <- read_counts(ds)
  if(is.null(d)) next
  t <- distinct(d,Signature,N_true)
  p_b <- ggplot(d,aes(N_RGS)) +
    geom_histogram(bins=25,fill="grey35",color="white") +
    geom_vline(data=t,aes(xintercept=N_true),
               color="red",linetype="dashed") +
    facet_wrap(~Signature,ncol=5,scales="free_x") +
    labs(title=ds,x="Number of retained RGSs",
         y="Number of label permutations") + theme_bw(base_size=9)
  print(p_b)
  ggsave(file.path(project_dir,"RGS_false_positive",
                   paste0("panel_b_",ds,".pdf")),p_b,width=14,height=11)
}
